# Advanced Colab Virtual Desktop - Complete Guide

This notebook demonstrates advanced usage of the **colab-virtual-desktop** package with full external access via ngrok tunneling.

**Features covered:**
- ✅ One-command setup with ngrok public URL
- ✅ Interactive control panel with ipywidgets
- ✅ Screenshot capture and display
- ✅ File transfer between Colab and desktop
- ✅ Session persistence
- ✅ Multiple applications management
- ✅ External browser access (URL works anywhere!)

## 📦 Installation & Setup

First, install the package and required system utilities:

In [ ]:
# Install the colab-virtual-desktop package
!pip install -q colab-virtual-desktop

# Install screenshot tools (scrot or imagemagick)
!apt-get update -qq
!apt-get install -y -qq scrot imagemagick > /dev/null 2>&1

print("✅ Installation complete!")

## 🔑 Get Your ngrok Token

You need an ngrok auth token to create a public URL. Get one free at:

**https://ngrok.com** → Sign up → Dashboard → "Your Authtoken"

### Option A: Store in Colab Secrets (Recommended)
1. Click **Runtime** → **Change runtime type** → **Secrets** tab
2. Add a new secret: `NGROK_AUTHTOKEN` = `your_token_here`

### Option B: Paste directly (less secure)
We'll ask you to paste it if not found in secrets.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Image, HTML, Javascript

# Try to get ngrok token from Colab secrets
NGROK_TOKEN = None
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
    if NGROK_TOKEN:
        print("✅ Found ngrok token in Colab secrets")
    else:
        print("⚠️  No NGROK_AUTHTOKEN found in secrets")
except Exception as e:
    print(f"ℹ️  Colab secrets not available: {e}")

# If not found, prompt user
if not NGROK_TOKEN:
    NGROK_TOKEN = input("Enter your ngrok auth token (from https://ngrok.com): ").strip()

if not NGROK_TOKEN:
    print("❌ No ngrok token provided. Cannot start desktop.")
    sys.exit(1)

print("\n🔑 Token ready. Starting desktop...")

## 🚀 Start the Virtual Desktop

We'll use the high-level `start_virtual_desktop()` helper function for one-command startup.

In [ ]:
from colab_desktop import start_virtual_desktop

# Start the desktop with one command!
desktop = start_virtual_desktop(
    ngrok_token=NGROK_TOKEN,
    geometry="1280x720",  # Resolution
    depth=24,              # Color depth
    vnc_password="colab123",  # Change for production!
    auto_open=False        # We'll display URL manually
)

# Get the public URL
desktop_url = desktop.get_url()

print(f"\n{'='*60}")
print(f"🎉 VIRTUAL DESKTOP IS RUNNING!")
print(f"{'='*60}")
print(f"🌐 Desktop URL: {desktop_url}")
print(f"📱 Copy this URL - it works from ANY browser!")
print(f"{'='*60}\n")

🚀 Setting up virtual desktop...
   Installing dependencies...
   This may take 1-2 minutes...
✅ Dependencies installed
   Setting up VNC password...
✅ VNC configured

▶️ Starting services...
   Starting Xvfb...
✅ Xvfb started
   Starting XFCE desktop...
✅ XFCE started
   Starting VNC server...
✅ VNC server started
   Starting noVNC web server...
✅ noVNC started
   Starting ngrok tunnel...
✅ ngrok tunnel created

✅ VIRTUAL DESKTOP READY!
📱 Desktop URL: https://f9c4-34-125-26-170.ngrok-free.app/vnc.html


## 🌐 Access Your Desktop

Click the link below to open your virtual desktop in a **new browser tab**:

In [ ]:
display(HTML(f'<a href="{desktop_url}" target="_blank" style="font-size: 24px; color: #4CAF50; font-weight: bold; padding: 20px; border: 2px solid #4CAF50; border-radius: 10px; text-decoration: none; display: inline-block;">🖥️ OPEN VIRTUAL DESKTOP</a>'))

🖥️ OPEN VIRTUAL DESKTOP

## 📦 Launch Applications

Let's launch some test applications to demonstrate the desktop:

In [ ]:
# Launch several X11 applications
apps = [
    ("xclock", "xclock &"),
    ("xcalc", "xcalc &"),
    ("xterm", "xterm &"),
    ("xeyes", "xeyes &"),
]

for name, cmd in apps:
    subprocess.Popen(cmd, shell=True)
    print(f"✅ Launched {name}")

print(f"📦 {len(apps)} applications running on virtual desktop!")

✅ Launched xclock
✅ Launched xcalc
✅ Launched xterm
✅ Launched xeyes
📦 4 applications running on virtual desktop!


## 📸 Take a Screenshot

We can capture the current desktop state and display it here in the notebook:

In [ ]:
def take_desktop_screenshot():
    """Take a screenshot of the virtual desktop"""
    screenshot_path = "/content/desktop_screenshot.png"
    
    print("📸 Taking screenshot...")
    
    # Try different screenshot methods
    methods = [
        f"DISPLAY={desktop.display} scrot {screenshot_path}",
        f"DISPLAY={desktop.display} import -window root {screenshot_path}",
        f"DISPLAY={desktop.display} xwd -out {screenshot_path}.xwd && convert {screenshot_path}.xwd {screenshot_path}",
    ]
    
    for cmd in methods:
        try:
            result = subprocess.run(cmd, shell=True, capture_output=True, timeout=5)
            if result.returncode == 0 and Path(screenshot_path).exists():
                print(f"Screenshot saved to: {screenshot_path}")
                return screenshot_path
        except:
            continue
    
    print("❌ Could not capture screenshot")
    return None

# Take and display screenshot
screenshot_file = take_desktop_screenshot()
if screenshot_file:
    display(Image(screenshot_file))
    print("✅ Screenshot captured!")
else:
    print("⚠️  Screenshot not available (scrot/imagemagick may not work in this environment)")

📸 Taking screenshot...
Screenshot saved to: /content/desktop_screenshot.png
✅ Screenshot captured!


## 🎛️ Interactive Control Panel

Create an interactive widget-based control panel to manage the desktop:

In [ ]:
out = widgets.Output()

button_start = widgets.Button(description="▶️ Start", button_style='success')
button_stop = widgets.Button(description="⏹️ Stop", button_style='danger')
button_restart = widgets.Button(description="🔄 Restart", button_style='warning')
button_screenshot = widgets.Button(description="📸 Screenshot", button_style='info')
url_label = widgets.HTML(value="<b>Status:</b> <span style='color: gray;'>Not started</span>")

def on_start(b):
    with out:
        clear_output()
        print("🚀 Starting desktop...")
        try:
            # Reset if needed
            if hasattr(desktop, 'is_running') and not desktop.is_running:
                desktop.start()
            print(f"✅ Desktop running at: {desktop.get_url()}")
            url_label.value = f"<b>Status:</b> <span style='color: green;'>Running</span> - <a href='{desktop.get_url()}' target='_blank'>Open Desktop</a>"
        except Exception as e:
            print(f"❌ Failed: {e}")
            url_label.value = f"<b>Status:</b> <span style='color: red;'>Error</span>"

def on_stop(b):
    with out:
        clear_output()
        desktop.stop()
        url_label.value = "<b>Status:</b> <span style='color: gray;'>Stopped</span>"
        print("🛑 Desktop stopped")

def on_restart(b):
    with out:
        clear_output()
        print("🔄 Restarting...")
        try:
            desktop.restart()
            print(f"✅ Desktop restarted: {desktop.get_url()}")
            url_label.value = f"<b>Status:</b> <span style='color: green;'>Running</span> - <a href='{desktop.get_url()}' target='_blank'>Open Desktop</a>"
        except Exception as e:
            print(f"❌ Restart failed: {e}")

def on_screenshot(b):
    with out:
        clear_output()
        print("📸 Capturing screenshot...")
        path = take_desktop_screenshot()
        if path:
            display(Image(path))
        else:
            print("❌ Failed")

button_start.on_click(on_start)
button_stop.on_click(on_stop)
button_restart.on_click(on_restart)
button_screenshot.on_click(on_screenshot)

control_panel = widgets.VBox([
    widgets.HTML("<h2>🖥️ Desktop Control Panel</h2>"),
    widgets.HBox([button_start, button_stop, button_restart, button_screenshot]),
    url_label,
    out
])

display(control_panel)

## 📁 File Transfer

Upload and download files between Colab and the virtual desktop:

In [ ]:
def upload_to_desktop(local_path: str, desktop_path: str = "/tmp/"):
    """Copy a file from Colab to the virtual desktop"""
    src = Path(local_path)
    if not src.exists():
        return f"❌ Error: {local_path} not found"
    
    dest = Path(desktop_path) / src.name
    import shutil
    shutil.copy2(local_path, dest)
    return f"✅ Copied {local_path} to {dest}"

def download_from_desktop(desktop_path: str, local_path: str = None):
    """Copy a file from virtual desktop to Colab"""
    src = Path(desktop_path)
    if not src.exists():
        return f"❌ Error: {desktop_path} not found"
    
    if local_path is None:
        local_path = Path("./") / src.name
    else:
        local_path = Path(local_path)
    
    import shutil
    shutil.copy2(src, local_path)
    return f"✅ Copied {desktop_path} to {local_path}"

# Demo: Create a file and upload it
test_file = Path("demo_upload.txt")
test_file.write_text("Hello from Colab Virtual Desktop!\nThis file was created in Colab and uploaded to the desktop.")

result = upload_to_desktop("demo_upload.txt", "/tmp/")
print(result)
print("\n💡 Tip: You can access this file on the desktop at /tmp/demo_upload.txt")

## 💾 Session Persistence

Save desktop state and restore it later:

In [ ]:
import pickle

def save_session(desktop, filename="/content/desktop_session.pkl"):
    """Save desktop configuration to file"""
    state = {
        'display': desktop.display,
        'geometry': desktop.geometry,
        'depth': desktop.depth,
        'vnc_port': desktop.vnc_port,
        'novnc_port': desktop.novnc_port,
        'vnc_password': desktop.vnc_password,
        'tunnel_url': desktop.tunnel_url,
        'is_running': desktop.is_running,
    }
    with open(filename, 'wb') as f:
        pickle.dump(state, f)
    print(f"✅ Session saved to {filename}")

def load_session(desktop, filename="/content/desktop_session.pkl"):
    """Load desktop configuration from file"""
    try:
        with open(filename, 'rb') as f:
            state = pickle.load(f)
        
        # Restore state
        desktop.display = state['display']
        desktop.geometry = state['geometry']
        desktop.depth = state['depth']
        desktop.vnc_port = state['vnc_port']
        desktop.novnc_port = state['novnc_port']
        desktop.vnc_password = state['vnc_password']
        desktop.tunnel_url = state['tunnel_url']
        
        print(f"✅ Session loaded from {filename}")
        return True
    except Exception as e:
        print(f"❌ Failed to load session: {e}")
        return False

# Save current session
save_session(desktop)
print("\n💾 You can save this file to Google Drive to persist across sessions.")

## 🐍 Run Python GUI Applications

Now let's run a Python GUI app on the desktop:

In [ ]:
import tkinter as tk

# Create a Tkinter app
root = tk.Tk()
root.title("Hello from Colab Virtual Desktop!")
root.geometry("400x250")

label = tk.Label(
    root,
    text="Welcome to\nColab Virtual Desktop!",
    font=("Arial", 16),
    pady=20
)
label.pack()

button = tk.Button(
    root,
    text="Close",
    command=root.destroy,
    font=("Arial", 12),
    bg="#4CAF50",
    fg="white",
    padx=20,
    pady=10
)
button.pack()

print("🚀 Launching Tkinter app on virtual desktop...")
root.mainloop()
print("✅ Tkinter app closed")

## 🎮 PyGame Example
Run a simple PyGame animation on the desktop:

In [ ]:
# Install pygame if needed
!pip install -q pygame

import pygame
import sys

# Initialize
pygame.init()
screen = pygame.display.set_mode((640, 480))
pygame.display.set_caption("PyGame on Colab Virtual Desktop!")

# Colors
WHITE = (255, 255, 255)
BLUE = (0, 120, 255)
RED = (255, 0, 0)

# Animation variables
clock = pygame.time.Clock()
counter = 0
running = True

print("🎮 Starting PyGame animation (will run for 5 seconds)...")

while running and counter < 300:  # 5 seconds at 60fps
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            running = False

    # Draw
    screen.fill(WHITE)
    x = 320 + int(150 * (counter / 300.0) * ((counter % 60) / 30.0 - 1))
    pygame.draw.circle(screen, BLUE, (x, 240), 40)

    # Text
    font = pygame.font.Font(None, 36)
    text = font.render(f"Frame: {counter}", True, RED)
    screen.blit(text, (10, 10))

    pygame.display.flip()
    clock.tick(60)
    counter += 1

pygame.quit()
print(f"✅ PyGame animation completed ({counter} frames)")

## 🧹 Cleanup

When you're done, stop the desktop to free Colab resources:

In [ ]:
print("🛑 Stopping virtual desktop...")
desktop.stop()
print("✅ Desktop stopped. Resources freed.")

# Also clear the session file if saved
session_file = Path("/content/desktop_session.pkl")
if session_file.exists():
    session_file.unlink()
    print("🗑️  Session file removed")

## 📋 Summary

✅ **You've learned how to:**
- Install and set up `colab-virtual-desktop`
- Get a public URL via ngrok tunneling
- Launch and control the virtual desktop
- Run multiple GUI applications simultaneously
- Take screenshots and display them in Colab
- Transfer files between Colab and desktop
- Save and load session state
- Run Python GUI apps (Tkinter, PyGame)
- Use an interactive control panel

### 🌟 Key Takeaways

1. **The desktop URL is public** - anyone with it can access your desktop
2. **Colab resets after ~12 hours** - you'll need to restart
3. **No persistence** - download important files before stopping
4. **Performance varies** - lower resolution for better speed
5. **Free tier works great** for experimentation and light use

### 🚀 Next Steps

- Try running your own GUI applications
- Experiment with different desktop environments
- Build and test GUI apps entirely in Colab
- Share the URL with collaborators (same ngrok tunnel)

Happy computing! 🎉